# 00 — Setup and smoke test
Run this after reopening the project in the VS Code Dev Container.

In [6]:
from pathlib import Path
import os, sys, subprocess, json
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
print(ROOT)

/workspace


In [7]:
from src.bootstrap import bootstrap_memory
pkg=bootstrap_memory()
print('memory:',pkg)

memory: /workspace/memory/trajdebug_flat_rag_86


In [ ]:
from dotenv import load_dotenv
load_dotenv(ROOT/'.env')
required=['AZURE_API_KEY','AZURE_API_BASE','AZURE_DEPLOYMENT']
missing=[k for k in required if not os.getenv(k)]
print('missing env vars:', missing)
print('deployment:', os.getenv('AZURE_DEPLOYMENT'))

In [9]:
print(subprocess.run(['docker','version','--format','{{.Server.Version}}'],capture_output=True,text=True).stdout.strip())
print(subprocess.run(['python','-c','import minisweagent; print(minisweagent.__version__)'],capture_output=True,text=True).stdout.strip())

29.8.0
This is mini-swe-agent version 2.4.6.
Check the v2 migration guide at https://klieret.short.gy/mini-v2-migration
Loading global config from '/root/.config/mini-swe-agent/.env'
2.4.6


In [10]:
from src.memory_conditions import SameInformationMemory
db=pkg/'data'/'flat_rag.sqlite'
q='legacy test expects old crash message but the requested behavior changed'
with SameInformationMemory(db) as m:
    a=m.flat(q, top_k=3, max_chars=6000)
    b=m.graph(q, seed_k=3, max_chars=6000, hops=1, max_neighbors=3, allowed_relations=['STATED_MOTIVATES','CITES_SOURCE_MESSAGE','QUALIFIED_BY','HAS_END_STATE_ASSESSMENT'])
print('flat selected',len(a.selected),'graph selected',len(b.selected))
print('graph traversal additions:', [x for x in b.selected if x['kind']=='graph_neighbor'][:3])

flat selected 2 graph selected 4
graph traversal additions: [{'kind': 'graph_neighbor', 'document_id': 'semantic|node|sem:C083:end_state', 'node_id': 'sem:C083:end_state', 'via_relation': 'HAS_END_STATE_ASSESSMENT', 'from_node': 'sem:C083:assessment'}, {'kind': 'graph_neighbor', 'document_id': 'semantic|node|sem:C083:limits', 'node_id': 'sem:C083:limits', 'via_relation': 'QUALIFIED_BY', 'from_node': 'sem:C083:assessment'}, {'kind': 'graph_neighbor', 'document_id': 'semantic|node|sem:C083:evidence:1', 'node_id': 'sem:C083:evidence:1', 'via_relation': 'CITES_SOURCE_MESSAGE', 'from_node': 'sem:C083:assessment'}]


If Docker, memory extraction, and Azure variables all look correct, continue to notebook 01. This notebook does **not** call the LLM.